# EU27 — Cluster-Informed Forecast (Pooled Prior + Shrinkage)

**What this notebook adds over the baseline forecast:**

Individual country trends for FEC in particular are noisy (27 of 27 countries have R² < 0.6).  
A linear trend fitted on 20 data points for a volatile series produces unreliable 2030 projections.

**Solution — James-Stein style shrinkage toward the cluster mean:**

For each country-dimension pair we blend the individual OLS slope with the cluster pooled slope,  
weighted by the reliability of the individual fit:

```
blended_slope = w · individual_slope + (1 − w) · cluster_slope
w = R²_individual   (shrinks noisy countries toward cluster, trusts clean fits)
```

This means:
- A country with R² = 0.95 uses almost entirely its own trend (w = 0.95)
- A country with R² = 0.10 leans heavily on its cluster's pooled trajectory (w = 0.10)

**Outputs:** `forecast_shrinkage.csv`, `shrinkage_comparison.png`, `divergence_flags.png`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ─────────────────────────────────────────────────────────────────────
PANEL_PATH    = "EU27_ESG_Panel_2005_2024.csv"
CLUSTER_PATH  = "EU27_cluster_labels_reduced.csv"
BASELINE_PATH = "forecast_results.csv"   # output from previous notebook
FORECAST_YEAR = 2030
CI_LEVEL      = 0.90
R2_THRESHOLD  = 0.60    # below this, flag for inspection

DARK_BG  = "#0f1117"
PANEL_BG = "#1a1d2e"
CLUSTER_COLORS = {1: "#ff6b6b", 2: "#00d4ff", 3: "#7fff7f", 4: "#ffd700"}
CLUSTER_NAMES  = {
    1: "Large Emitters",
    2: "Heterogeneous Middle",
    3: "Green Transition Leaders",
    4: "Germany (Singleton)",
}
TARGETS = {
    "GHG": {
        "actual":  "GHG_MtCO2eq",
        "target":  "GHG_FF55_Target_Mt",
        "label":   "GHG Emissions (MtCO₂eq)",
        "lower_is_better": True,
    },
    "FEC": {
        "actual":  "FEC_Mtoe",
        "target":  "FEC_FF55_Target_Mtoe",
        "label":   "Final Energy Consumption (Mtoe)",
        "lower_is_better": True,
    },
    "RES": {
        "actual":  "RES_Share_Pct",
        "target":  "RES_FF55_Target_Pct",
        "label":   "Renewable Energy Share (%)",
        "lower_is_better": False,
    },
}
# ──────────────────────────────────────────────────────────────────────────────
print("Config loaded ✓")


## 1. Load Data

In [ ]:
df       = pd.read_csv(PANEL_PATH)
clusters = pd.read_csv(CLUSTER_PATH)
baseline = pd.read_csv(BASELINE_PATH)

df = df.merge(clusters[["Country_Code","Cluster_k4"]], on="Country_Code")
df = df.sort_values(["Country_Code","Year"])

print(f"Panel   : {df.shape}")
print(f"Baseline: {baseline.shape}")


## 2. Step 1 — Fit Individual OLS Trends

Fit a linear trend per country per dimension. Record slope, intercept, R², and MAE.  
Countries below the R² threshold are flagged — these are the ones that will be shrunk most.


In [ ]:
individual_fits = {}   # key: (country_code, dim)

for code, grp in df.groupby("Country_Code"):
    grp = grp.sort_values("Year")
    X   = grp["Year"].values.reshape(-1, 1)
    cluster = grp["Cluster_k4"].iloc[0]

    for dim, cfg in TARGETS.items():
        y  = grp[cfg["actual"]].values
        lr = LinearRegression().fit(X, y)
        yp = lr.predict(X)
        r2 = max(r2_score(y, yp), 0.0)   # floor at 0 (no negative weights)

        individual_fits[(code, dim)] = {
            "slope":     lr.coef_[0],
            "intercept": lr.intercept_,
            "r2":        r2,
            "mae":       mean_absolute_error(y, yp),
            "cluster":   cluster,
            "target":    grp[cfg["target"]].iloc[0],
            "n":         len(y),
        }

# Summary of noisy series
rows = []
for (code, dim), fit in individual_fits.items():
    if fit["r2"] < R2_THRESHOLD:
        rows.append({"Country": code, "Dimension": dim, "R²": round(fit["r2"],3),
                     "Cluster": fit["cluster"]})

noisy = pd.DataFrame(rows).sort_values(["Dimension","R²"])
print(f"Series with R² < {R2_THRESHOLD}: {len(noisy)} of {27*3}")
noisy.style.background_gradient(subset=["R²"], cmap="RdYlGn").format({"R²": "{:.3f}"})


## 3. Step 2 — Fit Cluster Pooled Trends

For each cluster-dimension pair, normalise every country series to its 2005 baseline  
(so large and small countries contribute equally), then fit a single pooled OLS slope.


In [ ]:
cluster_fits = {}   # key: (cluster_id, dim)

for cluster_id in sorted(df["Cluster_k4"].unique()):
    cdf = df[df["Cluster_k4"] == cluster_id]

    for dim, cfg in TARGETS.items():
        rows = []
        for code, grp in cdf.groupby("Country_Code"):
            grp  = grp.sort_values("Year")
            base = grp[cfg["actual"]].iloc[0]
            if base == 0:
                continue
            for _, r in grp.iterrows():
                rows.append({"Year": r["Year"], "val_norm": r[cfg["actual"]] / base,
                             "country": code})

        pool = pd.DataFrame(rows)
        X    = pool["Year"].values.reshape(-1, 1)
        y    = pool["val_norm"].values
        lr   = LinearRegression().fit(X, y)

        cluster_fits[(cluster_id, dim)] = {
            "slope_norm":     lr.coef_[0],
            "intercept_norm": lr.intercept_,
            "r2":             round(r2_score(y, lr.predict(X)), 3),
        }

# Display
pool_df = pd.DataFrame([
    {"Cluster": f"{c}: {CLUSTER_NAMES[c]}", "Dimension": d,
     "Pooled Normalised Slope": round(v["slope_norm"], 5),
     "Pooled R²": v["r2"]}
    for (c, d), v in cluster_fits.items()
]).sort_values(["Dimension","Cluster"])

pool_df.style.background_gradient(subset=["Pooled R²"], cmap="RdYlGn").format(precision=4)


## 4. Step 3 — Shrinkage Blending

For each country-dimension, compute a blended slope:

```
w               = R²_individual  (reliability weight)
blended_slope   = w · individual_slope + (1−w) · cluster_slope_rescaled
```

The cluster slope (fitted on normalised data) is rescaled back to the country's  
own unit scale using its 2005 baseline value.


In [ ]:
PROJ_YEARS = np.arange(2005, 2031)

shrinkage_records = []

for code, grp in df.groupby("Country_Code"):
    grp     = grp.sort_values("Year")
    cluster = grp["Cluster_k4"].iloc[0]
    country = grp["Country_Name"].iloc[0]
    X       = grp["Year"].values.reshape(-1, 1)

    row = {"Country_Code": code, "Country": country,
           "Cluster": cluster, "Cluster_Name": CLUSTER_NAMES[cluster]}

    for dim, cfg in TARGETS.items():
        y          = grp[cfg["actual"]].values
        base_val   = y[0]
        target_val = grp[cfg["target"]].iloc[0]
        ind        = individual_fits[(code, dim)]
        pool       = cluster_fits[(cluster, dim)]

        # Rescale pooled slope from normalised → raw units
        cluster_slope_raw = pool["slope_norm"] * base_val

        # Shrinkage weight = individual R²
        w              = ind["r2"]
        blended_slope  = w * ind["slope"] + (1 - w) * cluster_slope_raw
        blended_interc = np.mean(y) - blended_slope * np.mean(grp["Year"].values)

        # Project to 2030
        proj = blended_interc + blended_slope * FORECAST_YEAR

        # Prediction interval using residuals from blended fit
        y_blended = blended_interc + blended_slope * grp["Year"].values
        resid     = y - y_blended
        s         = np.std(resid, ddof=2)
        n         = len(y)
        t_crit    = stats.t.ppf((1 + CI_LEVEL) / 2, df=n - 2)
        margin    = t_crit * s * np.sqrt(1 + 1/n)

        lib       = cfg["lower_is_better"]
        on_track  = proj <= target_val if lib else proj >= target_val

        # Pull baseline individual projection for comparison
        ind_proj = ind["intercept"] + ind["slope"] * FORECAST_YEAR

        row.update({
            f"{dim}_ind_proj":       round(ind_proj, 3),
            f"{dim}_blended_proj":   round(proj, 3),
            f"{dim}_ci_lo":          round(proj - margin, 3),
            f"{dim}_ci_hi":          round(proj + margin, 3),
            f"{dim}_target":         round(target_val, 3),
            f"{dim}_gap":            round(proj - target_val, 3),
            f"{dim}_on_track":       on_track,
            f"{dim}_w":              round(w, 3),
            f"{dim}_ind_r2":         round(ind["r2"], 3),
            f"{dim}_proj_shift":     round(proj - ind_proj, 3),  # how much shrinkage moved it
        })

    shrinkage_records.append(row)

results = pd.DataFrame(shrinkage_records).sort_values(["Cluster","Country"])
print(f"Shrinkage forecasts computed for {len(results)} countries ✓")


## 5. Results — Blended Projections vs Baseline

In [ ]:
def style_bool(v):
    if v is True:  return "background-color: #1a3d1a; color: #7fff7f"
    if v is False: return "background-color: #3d1a1a; color: #ff6b6b"
    return ""

summary_cols = ["Country","Cluster_Name",
                "GHG_blended_proj","GHG_target","GHG_on_track","GHG_w",
                "FEC_blended_proj","FEC_target","FEC_on_track","FEC_w",
                "RES_blended_proj","RES_target","RES_on_track","RES_w"]

results[summary_cols].style \
    .applymap(style_bool, subset=["GHG_on_track","FEC_on_track","RES_on_track"]) \
    .format(precision=2) \
    .set_caption("w = shrinkage weight (= individual R²). Low w → heavily influenced by cluster prior.")


## 6. Divergence Flags

Countries where shrinkage moved the 2030 projection by more than ±5% of the target value.  
These are the most analytically interesting cases — either the individual trend was  
misleading, or the country is genuinely breaking away from its cluster trajectory.


In [ ]:
flag_rows = []
for _, r in results.iterrows():
    for dim in ["GHG","FEC","RES"]:
        shift    = abs(r[f"{dim}_proj_shift"])
        target   = abs(r[f"{dim}_target"])
        pct_move = shift / target * 100 if target != 0 else 0
        if pct_move > 5:
            flag_rows.append({
                "Country":      r["Country"],
                "Cluster":      r["Cluster_Name"],
                "Dimension":    dim,
                "Ind. Proj":    r[f"{dim}_ind_proj"],
                "Blended Proj": r[f"{dim}_blended_proj"],
                "Target":       r[f"{dim}_target"],
                "Shift":        round(r[f"{dim}_proj_shift"], 2),
                "% Move":       round(pct_move, 1),
                "Ind. R²":      r[f"{dim}_ind_r2"],
                "Weight (w)":   r[f"{dim}_w"],
            })

flags = pd.DataFrame(flag_rows).sort_values("% Move", ascending=False)
print(f"{len(flags)} divergence flags (shift > 5% of target)")
flags.style \
    .background_gradient(subset=["% Move"], cmap="YlOrRd") \
    .background_gradient(subset=["Weight (w)"], cmap="RdYlGn") \
    .format(precision=2)


## 7. Comparison Plot — Individual vs Blended Projections

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.patch.set_facecolor(DARK_BG)

for ax, dim in zip(axes, ["GHG","FEC","RES"]):
    ax.set_facecolor(PANEL_BG)
    cfg = TARGETS[dim]

    for _, r in results.iterrows():
        cluster = r["Cluster"]
        c_color = CLUSTER_COLORS[cluster]
        ind_p   = r[f"{dim}_ind_proj"]
        bld_p   = r[f"{dim}_blended_proj"]
        target  = r[f"{dim}_target"]
        w       = r[f"{dim}_w"]

        # Arrow from individual → blended
        ax.annotate("",
            xy=(bld_p, r.name), xytext=(ind_p, r.name),
            arrowprops=dict(arrowstyle="->", color=c_color, lw=1.2, alpha=0.7))

        # Blended dot
        on_track = r[f"{dim}_on_track"]
        dot_color = "#7fff7f" if on_track else "#ff6b6b"
        ax.scatter(bld_p, r.name, color=dot_color, s=50, zorder=5)

        # Target line per country
        ax.scatter(target, r.name, marker="|", color="#ffd700", s=80, zorder=4, alpha=0.7)

    ax.set_yticks(range(len(results)))
    ax.set_yticklabels(results["Country"].values, fontsize=7, color="white")
    ax.set_title(f"{dim} — {cfg['label']}", color="white", fontsize=11, fontweight="bold", pad=10)
    ax.set_xlabel("Projected 2030 value", color="#aaa", fontsize=9)
    ax.tick_params(colors="#aaa")
    ax.spines[:].set_edgecolor("#333")

    # Legend
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    legend_els = [
        Line2D([0],[0], color="gray", marker=">", label="arrow: ind → blended"),
        Line2D([0],[0], color="#ffd700", marker="|", linestyle="None", label="2030 target", markersize=10),
        Line2D([0],[0], color="#7fff7f", marker="o", linestyle="None", label="on track"),
        Line2D([0],[0], color="#ff6b6b", marker="o", linestyle="None", label="at risk"),
    ]
    ax.legend(handles=legend_els, fontsize=7, facecolor=PANEL_BG,
              labelcolor="white", loc="lower right")

    # Cluster colour band on right
    for i, (_, r) in enumerate(results.iterrows()):
        ax.add_patch(plt.Rectangle(
            (ax.get_xlim()[1] if ax.get_xlim()[1] != 1.0 else 0, i - 0.4),
            1, 0.8, color=CLUSTER_COLORS[r["Cluster"]], clip_on=False, alpha=0.6))

plt.suptitle("Individual vs Shrinkage-Blended 2030 Projections\n"
             "(arrows show how cluster prior shifted the forecast)",
             color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("shrinkage_comparison.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()
print("Saved → shrinkage_comparison.png")


## 8. Cluster-Level On-Track Summary (Blended)

In [ ]:
summary = results.groupby(["Cluster","Cluster_Name"]).agg(
    Countries         = ("Country_Code", "count"),
    GHG_on_track      = ("GHG_on_track", "sum"),
    FEC_on_track      = ("FEC_on_track", "sum"),
    RES_on_track      = ("RES_on_track", "sum"),
    GHG_avg_gap       = ("GHG_gap", "mean"),
    FEC_avg_gap       = ("FEC_gap", "mean"),
    RES_avg_gap       = ("RES_gap", "mean"),
    GHG_avg_w         = ("GHG_w", "mean"),
    FEC_avg_w         = ("FEC_w", "mean"),
    RES_avg_w         = ("RES_w", "mean"),
).reset_index()

for dim in ["GHG","FEC","RES"]:
    summary[f"{dim}_%"] = (summary[f"{dim}_on_track"] / summary["Countries"] * 100).round(0).astype(int).astype(str) + "%"

display_cols = ["Cluster_Name","Countries",
                "GHG_on_track","GHG_%","GHG_avg_gap","GHG_avg_w",
                "FEC_on_track","FEC_%","FEC_avg_gap","FEC_avg_w",
                "RES_on_track","RES_%","RES_avg_gap","RES_avg_w"]
summary[display_cols].style.format(precision=2)


## 9. Export

In [ ]:
export_cols = [
    "Country_Code","Country","Cluster","Cluster_Name",
    "GHG_ind_proj","GHG_blended_proj","GHG_ci_lo","GHG_ci_hi",
    "GHG_target","GHG_gap","GHG_on_track","GHG_ind_r2","GHG_w","GHG_proj_shift",
    "FEC_ind_proj","FEC_blended_proj","FEC_ci_lo","FEC_ci_hi",
    "FEC_target","FEC_gap","FEC_on_track","FEC_ind_r2","FEC_w","FEC_proj_shift",
    "RES_ind_proj","RES_blended_proj","RES_ci_lo","RES_ci_hi",
    "RES_target","RES_gap","RES_on_track","RES_ind_r2","RES_w","RES_proj_shift",
]
results[export_cols].to_csv("forecast_shrinkage.csv", index=False)
print("Saved → forecast_shrinkage.csv")
results[export_cols]
